In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)...",  # Change as you preferred
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)...",
]

BAR_VALUE_MAP = {"40.png": 4, "60.png": 3, "80.png": 2}

def count_snowflakes(td_element):
    score = 0.0
    for img in td_element.find_all('img'):
        src = img.get('src', '').lower()
        if "half" in src:
            score += 0.5
        elif "snowflake" in src:
            score += 1.0
    return score


def safe_get(session, url, retries=3):
    for i in range(retries):
        try:
            res = session.get(url, timeout=10)
            if res.status_code == 200:
                return res
        except requests.exceptions.RequestException:
            time.sleep(2)
    return None


def main():
    api_url = "https://thegoodride.com/ajax.php?mens=0&womens=0"

    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://thegoodride.com/",
    }

    # Session
    session = requests.Session()
    session.headers.update(headers)

    print("Fetching snowboard list from API...")
    res = safe_get(session, api_url)

    if not res:
        print("API request failed.")
        return


    # 只取前100个
    all_boards_data = res.json()
    all_boards_data = all_boards_data[:100]
    print(f"Fetched {len(all_boards_data)} boards.")

    results = []

    for board in all_boards_data:
        try:
            brand = board[0] if len(board) > 0 else "N/A"
            name = board[1] if len(board) > 1 else "N/A"
            price_str = board[3] if len(board) > 3 else "0"
            board_id = board[4] if len(board) > 4 else None

            if not board_id:
                continue

            usd_price = int("".join(re.findall(r'\d+', str(price_str))) or 0)

            clean_name = re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '-')
            clean_brand = re.sub(r'[^\w\s-]', '', brand).strip().replace(' ', '-')

            detail_url = f"https://thegoodride.com/snowboard-reviews/{clean_brand}-{clean_name}-{board_id}/"

            print(f"分析中: {brand} {name}...")

            d_res = safe_get(session, detail_url)
            if not d_res:
                print("Detail page failed")
                continue

            soup = BeautifulSoup(d_res.text, 'html.parser')

            specs = {}

            for row in soup.find_all('tr'):
                label = row.find('span', class_='rating-align-left')

                if label:
                    td_cells = row.find_all('td')
                    if len(td_cells) > 1:
                        specs[label.get_text(strip=True)] = count_snowflakes(td_cells[1])

                if row.td and "Flex" in row.td.get_text():
                    img = row.find('div', class_='rating-wrapper')
                    if img:
                        img = img.find('img')
                        if img:
                            m = re.search(r'(\d+)\.png', img['src'])
                            if m:
                                specs['Flex_Val'] = BAR_VALUE_MAP.get(m.group(0), 3)

            pw = specs.get('Powder', 0)
            cv = specs.get('Carving', 0)
            fs = (specs.get('Buttering', 0) +
                  specs.get('Jumps', 0) +
                  specs.get('Flex_Val', 3)) / 3

            value_deal = round((pw + cv + fs) / (usd_price / 100), 2) if usd_price > 0 else 0

            results.append({
                "brand": brand,
                "name": name,
                "price": usd_price,
                "value": value_deal
            })

            time.sleep(random.uniform(1, 2.5))

        except Exception as e:
            print("Error:", e)

    # 排序
    results.sort(key=lambda x: x['value'], reverse=True)

    output_file = "snowboards_top100.json"

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"\nDone！共处理 {len(results)} 个雪板")
    print(f"结果保存在: {output_file}")


if __name__ == "__main__":
    main()

Fetching snowboard list from API...
Fetched 100 boards.
分析中: Telos  BackSlash...
分析中: K2 Excavator...
分析中: Stranda Makrill...
分析中: Capita Spring Break Stairmaster...
